In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Typical spam vs ham examples
spam_examples = [
    "URGENT! You've won $1000000! Click here NOW!",
    "FREE VIAGRA! No prescription needed! Call 555-SPAM",
    "Congratulations! You're our 1000th visitor! Claim your prize!",
    "Make $5000/week working from home! No experience required!",
    "STOP! Don't pay your credit card bill until you read this!"
]

ham_examples = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Your package has been delivered to your front door.",
    "Meeting rescheduled to 3 PM. See you in conference room B.",
    "Thanks for the great presentation today!",
    "Your subscription renewal is due next month."
]


In [3]:
# Typical spam vs ham examples in Thai
spam_examples = [
    "ด่วน! คุณชนะรางวัล 1,000,000 บาท! คลิกที่นี่เลย!",
    "ฟรี! ยาเพิ่มสมรรถภาพ! ไม่ต้องใบสั่งแพทย์! โทร 555-SPAM",
    "ยินดีด้วย! คุณเป็นผู้เข้าชมคนที่ 1000! รับรางวัลของคุณ!",
    "หารายได้ 5,000 บาท/สัปดาห์ ทำงานที่บ้าน! ไม่ต้องมีประสบการณ์!",
    "หยุด! อย่าจ่ายบัตรเครดิตจนกว่าคุณจะอ่านข้อความนี้!"
]

ham_examples = [
    "เฮ้ เรายังนัดกินข้าวเที่ยงพรุ่งนี้อยู่ใช่ไหม?",
    "พัสดุของคุณถูกส่งถึงหน้าประตูบ้านแล้ว",
    "ประชุมเลื่อนเป็นบ่าย 3 โมง พบกันที่ห้องประชุม B",
    "ขอบคุณสำหรับการนำเสนอที่ยอดเยี่ยมวันนี้!",
    "การต่ออายุสมาชิกของคุณครบกำหนดเดือนหน้า"
]


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# 1. Prepare data
data = {
    'text': spam_examples + ham_examples,
    'labels': [1]*len(spam_examples) + [0]*len(ham_examples)  # 1=spam, 0=ham
}
dataset = Dataset.from_dict(data)

# 2. Load model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# 2. Load model - Using Qwen2.5-0.5B
# tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
# model = AutoModelForSequenceClassification.from_pretrained('Qwen/Qwen2.5-0.5B', num_labels=2)


# 3. Tokenize
def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

dataset = dataset.map(tokenize, batched=True)

# 4. Train (minimal setup)
training_args = TrainingArguments(
    output_dir='./artifacts/spam_model',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
    # tokenizer=tokenizer,
)

trainer.train()  # Uncomment to train

d:\fine-tune-text-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11750.40it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were 

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.22it/s]


TrainOutput(global_step=6, training_loss=0.6091585159301758, metrics={'train_runtime': 0.958, 'train_samples_per_second': 31.315, 'train_steps_per_second': 6.263, 'total_flos': 108664662960.0, 'train_loss': 0.6091585159301758, 'epoch': 3.0})

In [5]:
model.save_pretrained('./artifacts/spam_model')
tokenizer.save_pretrained('./artifacts/spam_model')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.06it/s]


('./artifacts/spam_model\\tokenizer_config.json',
 './artifacts/spam_model\\tokenizer.json')

In [24]:
# Predict on new text
test_texts = [
    "WIN FREE MONEY NOW!!!",
    "Let's grab coffee tomorrow"
]

test_texts = [
    "ชนะเงินฟรีตอนนี้!!!",
    "ไปดื่มกาแฟกันพรุ่งนี้นะ"
]

device = model.device
inputs = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt").to(device)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)

for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'}\n")


Text: ชนะเงินฟรีตอนนี้!!!
Prediction: SPAM

Text: ไปดื่มกาแฟกันพรุ่งนี้นะ
Prediction: HAM



In [16]:
pred

tensor(0, device='cuda:0')

In [17]:
# Load saved model
tokenizer = AutoTokenizer.from_pretrained('./artifacts/spam_model')
model = AutoModelForSequenceClassification.from_pretrained('./artifacts/spam_model')

# Predict
# test_texts = ["WIN FREE MONEY NOW!!!", "Let's grab coffee tomorrow"]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
inputs = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt").to(device)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)

for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'}\n")


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11848.64it/s]

Text: ชนะเงินฟรีตอนนี้!!!
Prediction: SPAM

Text: ไปดื่มกาแฟกันพรุ่งนี้นะ
Prediction: HAM



In [18]:
pred

tensor(0, device='cuda:0')